# 02. quotes: 시세 파일 → {날짜: {키: 값}}

`src/quanttool/market/quotes.py`

| 의존하는 것 | 제공하는 것 |
|---|---|
| 없음 (표준 라이브러리만) | `read_quotes` → 날짜별 시세 사전. `build_market`이 하루치를 받는다 |

**객체 지도** (이 순서로 본다)

1. 파일 형식
2. `read_quotes` 반환 구조
3. 여러 파일·구분자·주석
4. 오류

> 데이터는 `tests/fixtures/`의 **합성 샘플**이다. 구조를 보는 용도이고 숫자로 시장을 해석하지 않는다.

In [ ]:
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
FIX = ROOT / "tests" / "fixtures"  # 합성 샘플 시세·예시 북
ASOF = date(2026, 10, 23)

from quanttool.market.quotes import read_quotes

print("".join((FIX / "market_sample.txt").read_text().splitlines(True)[:6]))

## `read_quotes(paths)` → `dict[date, dict[str, float]]`

In [ ]:
quotes = read_quotes(FIX / "market_sample.txt")
print(type(quotes).__name__, list(quotes))
q = quotes[ASOF]
print(len(q), "개 키, 예:", list(q.items())[:3])
pd.DataFrame(quotes).head(8)  # 행: 키, 열: 날짜

키의 모양은 `종류/측정/통화/이름/만기`. 커브 레시피의 `quote_key` 틀이 이 키를 만든다.

In [ ]:
sorted({k.rsplit("/", 1)[0] for k in q})

## 여러 파일, 구분자(공백·쉼표·탭), `#` 주석

In [ ]:
import tempfile

tmp = Path(tempfile.mkdtemp())
(tmp / "a.txt").write_text("# 주석\n2026-10-23,X/1Y,0.01\n")
(tmp / "b.txt").write_text("2026-10-23\tX/2Y\t0.02   # 줄 끝 주석\n")
read_quotes([tmp / "a.txt", tmp / "b.txt"])

## 오류: 같은 날 같은 키 두 번, 형식이 `날짜 키 값`이 아님

In [ ]:
(tmp / "dup.txt").write_text("2026-10-23 X/1Y 0.01\n2026-10-23 X/1Y 0.02\n")
(tmp / "bad.txt").write_text("2026-10-23 X/1Y\n")
for f in ["dup.txt", "bad.txt"]:
    try:
        read_quotes(tmp / f)
    except ValueError as e:
        print(e)

## 요약

| 객체 | 입력 | 출력 | 언제 쓰나 |
|---|---|---|---|
| `read_quotes(paths)` | 파일 경로 하나 또는 여러 개 | `{date: {키: 값}}` (날짜순) | 시장을 만들기 전, 하루치를 꺼내 `build_market`에 |

## 내 메모

- 